# Is this message flirty? 💬

A compact, reproducible baseline for classifying **one private-chat message** as flirty or neutral. The training data contains English messages and synthetic Russian and Kazakh translations. A character-based model also accepts mixed text, spelling variation, slang, and emoji.

> **Important limitation:** an isolated message cannot reveal a sender's intent with certainty. This educational model recognizes textual patterns, not feelings or consent.

## 1. Setup and data loading

We preserve punctuation, casing, and emoji because they can be useful signals. The supplied train, validation, and test split is kept intact.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
DATA_DIR = Path("data") if Path("data/train.csv").is_file() else Path(".")
SPLITS = {
    name: pd.read_csv(DATA_DIR / f"{name}.csv")
    for name in ("train", "validation", "test")
}
train, validation, test = (SPLITS[name] for name in SPLITS)

## 2. Data quality audit

The checks below fail early if a file is malformed. Each source message must occur exactly once in each language, with the same label, and source IDs must not cross splits.

In [ ]:
EXPECTED_COLUMNS = {"id", "label", "texts", "language"}
EXPECTED_LANGUAGES = {"en", "ru", "kk"}

for name, frame in SPLITS.items():
    assert set(frame.columns) == EXPECTED_COLUMNS, f"Unexpected columns in {name}"
    assert not frame.isna().any().any(), f"Missing values in {name}"
    assert set(frame["label"].unique()) == {0, 1}, f"Bad labels in {name}"
    assert set(frame["language"].unique()) == EXPECTED_LANGUAGES
    grouped = frame.groupby("id")
    assert grouped.size().eq(3).all(), f"Every {name} ID must have three rows"
    assert grouped["language"].nunique().eq(3).all(), f"Missing translation in {name}"
    assert grouped["label"].nunique().eq(1).all(), f"Label mismatch in {name}"

id_sets = {name: set(frame["id"]) for name, frame in SPLITS.items()}
assert id_sets["train"].isdisjoint(id_sets["validation"])
assert id_sets["train"].isdisjoint(id_sets["test"])
assert id_sets["validation"].isdisjoint(id_sets["test"])

audit = pd.DataFrame({
    name: {
        "rows": len(frame),
        "source messages": frame["id"].nunique(),
        "flirty": int(frame["label"].sum()),
        "neutral": int((frame["label"] == 0).sum()),
        "missing": int(frame.isna().sum().sum()),
    }
    for name, frame in SPLITS.items()
}).T
display(audit)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

(train.groupby(["language", "label"]).size().unstack()
 .rename(columns={0: "neutral", 1: "flirty"})
 .plot.bar(ax=axes[0], color=["#64748b", "#ec4899"], rot=0))
axes[0].set(title="Training examples by language", xlabel="Language", ylabel="Messages")
axes[0].legend(frameon=False)

for language, group in train.groupby("language"):
    axes[1].hist(group["texts"].str.len(), bins=30, alpha=0.55, label=language)
axes[1].set(title="Message-length distribution", xlabel="Characters", ylabel="Messages", xlim=(0, 250))
axes[1].legend(frameon=False)
plt.tight_layout()

### Ambiguity and duplicate leakage

Identical short messages can reasonably receive different labels without conversational context. We report those conflicts instead of silently deleting them. We also measure normalized text overlap across splits; ID leakage is already forbidden above.

In [ ]:
def normalized_text(series):
    return series.str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()

train_audit = train.assign(normalized=normalized_text(train["texts"]))
conflicts = (
    train_audit.groupby(["language", "normalized"])["label"]
    .agg(["size", "nunique"])
    .query("nunique > 1")
    .reset_index()
)
display(conflicts)

overlaps = []
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    for language in sorted(EXPECTED_LANGUAGES):
        left_text = set(normalized_text(SPLITS[left].query("language == @language")["texts"]))
        right_text = set(normalized_text(SPLITS[right].query("language == @language")["texts"]))
        overlaps.append({"splits": f"{left} ↔ {right}", "language": language, "overlap": len(left_text & right_text)})
display(pd.DataFrame(overlaps).pivot(index="splits", columns="language", values="overlap"))

## 3. Character TF–IDF baseline

Character n-grams learn short fragments rather than whole words. This is a strong small-data baseline for chat because it tolerates typos, slang, inflections, mixed alphabets, and emoji. Logistic regression is fast and gives probabilities. Model choices are made using validation data only.

In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
])
model.fit(train["texts"], train["label"])

validation_predictions = model.predict(validation["texts"])
print(f"Validation macro-F1: {f1_score(validation['label'], validation_predictions, average='macro'):.3f}")
print(classification_report(validation["label"], validation_predictions, target_names=["neutral", "flirty"]))

## 4. Final test evaluation

The test set is used once after choosing the baseline. Aggregate performance can hide a weak language, so results are shown separately for English, Russian, and Kazakh.

In [ ]:
test_predictions = model.predict(test["texts"])
test_results = test.assign(prediction=test_predictions)

scores = []
for language, group in test_results.groupby("language"):
    scores.append({
        "language": language,
        "messages": len(group),
        "macro-F1": f1_score(group["label"], group["prediction"], average="macro"),
    })
scores.append({
    "language": "all",
    "messages": len(test_results),
    "macro-F1": f1_score(test_results["label"], test_results["prediction"], average="macro"),
})
display(pd.DataFrame(scores).set_index("language").round(3))

ConfusionMatrixDisplay.from_predictions(
    test["label"], test_predictions, display_labels=["neutral", "flirty"], cmap="RdPu"
)
plt.title("Test-set confusion matrix")
plt.show()

## 5. Error analysis

The most useful mistakes are confident wrong predictions. Many reveal ambiguous labels, missing context, or translation artifacts rather than a simple modeling error.

In [ ]:
flirty_probability = model.predict_proba(test["texts"])[:, 1]
errors = test.assign(
    prediction=test_predictions,
    flirty_probability=flirty_probability,
).query("label != prediction").copy()
errors["confidence"] = np.maximum(errors["flirty_probability"], 1 - errors["flirty_probability"])
display(
    errors.sort_values("confidence", ascending=False)
    [["language", "texts", "label", "prediction", "flirty_probability"]]
    .head(20)
)

## 6. Try your own messages

Probabilities near 50% are shown as uncertain. This is a presentation choice, not proof of the sender's intent. The model was trained only on English plus synthetic Russian and Kazakh, so other languages are out of scope.

In [ ]:
def classify_message(message):
    if not isinstance(message, str) or not message.strip():
        raise ValueError("message must be a non-empty string")
    probability = model.predict_proba([message])[0, 1]
    label = "uncertain" if 0.4 <= probability <= 0.6 else ("flirty" if probability > 0.6 else "neutral")
    return {"message": message, "prediction": label, "flirty probability": round(float(probability), 3)}

examples = [
    "How was your day?",
    "I can't stop thinking about your smile 😉",
    "Ты сегодня прекрасно выглядишь ❤️",
    "Сәлем, coffee ішеміз бе? 😊",
]
pd.DataFrame(classify_message(message) for message in examples)

## Conclusion

This notebook provides a transparent baseline, not a mind reader. Its main limitations are the small dataset, contradictory labels, lack of conversation context, and machine-translated Russian and Kazakh. A serious next step would be collecting native, contextual conversations—not adding a larger model.